In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com",
    timeout=300
)

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [13]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [14]:
search_tool = {
    "type": "function",
    "function": {  
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [15]:
response = openai_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[search_tool]
)

In [22]:
def agent_loop(instructions, question, model="deepseek-v4-flash") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        messages.append(response.choices[0].message)

        for item in response.choices[0].message:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [25]:
import json

def make_call(tool_call):
    """Handle function call for DeepSeek"""
    args = json.loads(tool_call.function.arguments)
    
    if tool_call.function.name == "search":
        result = search(**args)
    else:
        result = {"error": f"Unknown function: {tool_call.function.name}"}
    
    result_json = json.dumps(result, ensure_ascii=False, indent=2)
    

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json
    }

def agent_loop(instructions, question, model="deepseek-v4-flash") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool],
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message
        
        messages.append({
            "role": "assistant",
            "content": assistant_message.content,
            "tool_calls": assistant_message.tool_calls
        })

        if assistant_message.tool_calls:
            has_function_calls = True
            
            for tool_call in assistant_message.tool_calls:
                print(f"function_call: {tool_call.function.name} {tool_call.function.arguments}")
                
                call_output = make_call(tool_call)
                messages.append(call_output)
        
        else:
            if assistant_message.content:
                print("ASSISTANT:")
                last_answer = assistant_message.content
                print(last_answer)
            else:
                print("ASSISTANT: (empty response)")

        it += 1
        
        if has_function_calls == False:
            break

    return last_answer

In [26]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query": "Olama run locally"}
iteration #2...
function_call: search {"query": "Ollama run locally"}
iteration #3...
ASSISTANT:
I believe you meant **Ollama** (a popular tool for running LLMs locally) — and I've found some great resources for you! Here's a step-by-step guide:

---

## 🚀 How to Install & Run Ollama Locally

### 1. Install Ollama

Visit **[https://ollama.com/download](https://ollama.com/download)** and choose your OS:

| OS | Command / Instructions |
|---|---|
| **macOS** | Download the `.pkg` and install it. |
| **Windows** | Download the `.msi` and install it. |
| **Linux** | Run in the terminal: `curl -fsSL https://ollama.com/install.sh | sh` |

### 2. Run a Model

Open a terminal and type:

```bash
ollama run llama3
```

This will:
- Download the **LLaMA 3** model (~4GB on first run).
- Start the model locally.
- Open an interactive chat interface where you can type questions directly.

### 3. Test the Local Server

Ollama runs a

'I believe you meant **Ollama** (a popular tool for running LLMs locally) — and I\'ve found some great resources for you! Here\'s a step-by-step guide:\n\n---\n\n## 🚀 How to Install & Run Ollama Locally\n\n### 1. Install Ollama\n\nVisit **[https://ollama.com/download](https://ollama.com/download)** and choose your OS:\n\n| OS | Command / Instructions |\n|---|---|\n| **macOS** | Download the `.pkg` and install it. |\n| **Windows** | Download the `.msi` and install it. |\n| **Linux** | Run in the terminal: `curl -fsSL https://ollama.com/install.sh | sh` |\n\n### 2. Run a Model\n\nOpen a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis will:\n- Download the **LLaMA 3** model (~4GB on first run).\n- Start the model locally.\n- Open an interactive chat interface where you can type questions directly.\n\n### 3. Test the Local Server\n\nOllama runs a local API server on port **11434**. You can test it with:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a resp

In [27]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query": "Can I still join the course late"}
iteration #2...
ASSISTANT:
Great question! Here's what I found from the course FAQ:

**Yes, you can still join!** 🎉

However, there are a couple of things to keep in mind:

1. **If you're aiming for a certificate**: You need to submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort (not self-paced mode), which involves peer-reviewing 3 capstone projects after submitting your own.

2. **Homework is optional**: Even if you missed earlier homeworks, you can still earn a certificate by passing the Capstone project. (Homework is recommended for practice but not mandatory for the certificate.)

3. **Registration is not required**: You don't need to be registered to start learning or submitting work. You can simply dive in and follow the materials.

---

**Quick summary:**
- ✅ You can join anytime
- ✅ You can still get

'Great question! Here\'s what I found from the course FAQ:\n\n**Yes, you can still join!** 🎉\n\nHowever, there are a couple of things to keep in mind:\n\n1. **If you\'re aiming for a certificate**: You need to submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort (not self-paced mode), which involves peer-reviewing 3 capstone projects after submitting your own.\n\n2. **Homework is optional**: Even if you missed earlier homeworks, you can still earn a certificate by passing the Capstone project. (Homework is recommended for practice but not mandatory for the certificate.)\n\n3. **Registration is not required**: You don\'t need to be registered to start learning or submitting work. You can simply dive in and follow the materials.\n\n---\n\n**Quick summary:**\n- ✅ You can join anytime\n- ✅ You can still get a certificate if you submit your project in time\n- ✅ No registration needed — just start 

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query": "joining course late enrollment"}
function_call: search {"query": "can I join the course"}
iteration #2...
ASSISTANT:
Great news! Here's what I found:

**Yes, you can still join!** 🎉

The FAQ directly states: *"I just discovered the course. Can I still join?"* — **Yes!**

However, there are a couple of things to keep in mind:

### 📜 Regarding a Certificate
If you're interested in earning a **certificate**, you need to complete the course with a **"live" cohort** and submit your project **while submissions are still being accepted**. Certificates are not awarded for self-paced mode (since you need to peer-review 3 capstone projects, which only happens while the course is running).

### 📋 Registration vs. Participation
You mentioned you just discovered the course — but you don't actually need to register to start learning! You can simply begin accessing the materials and submitting homework. Registration is just to gauge interest before the

'Great news! Here\'s what I found:\n\n**Yes, you can still join!** 🎉\n\nThe FAQ directly states: *"I just discovered the course. Can I still join?"* — **Yes!**\n\nHowever, there are a couple of things to keep in mind:\n\n### 📜 Regarding a Certificate\nIf you\'re interested in earning a **certificate**, you need to complete the course with a **"live" cohort** and submit your project **while submissions are still being accepted**. Certificates are not awarded for self-paced mode (since you need to peer-review 3 capstone projects, which only happens while the course is running).\n\n### 📋 Registration vs. Participation\nYou mentioned you just discovered the course — but you don\'t actually need to register to start learning! You can simply begin accessing the materials and submitting homework. Registration is just to gauge interest before the start date.\n\n### ✅ Homework\nEven if you\'ve missed some homework deadlines, that\'s okay! Homework is **not mandatory** for getting a certificate 

In [29]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query": "Queen's Gambit"}
iteration #2...
function_call: search {"query": "queen gambit chess opening"}
function_call: search {"query": "gambit"}
iteration #3...
function_call: search {"query": "queen's gambit chess"}
iteration #4...
ASSISTANT:
It looks like the **Queen's Gambit** isn't covered in our course FAQ materials. However, I can answer based on general knowledge!

The **Queen's Gambit** is a classic chess opening that begins with the moves:

1. **d4 d5**
2. **c4**

Here's a quick breakdown:

- **The idea:** White offers a pawn (the one on c4) to Black in exchange for better control of the center of the board. If Black accepts the pawn (the **Queen's Gambit Accepted**), White can try to gain a positional advantage. If Black declines (the **Queen's Gambit Declined** — usually with **e6**), the game tends to be more closed and strategic.
- **It's not a "true" gambit:** In most cases, White can safely regain the pawn if needed, so it's more 

'It looks like the **Queen\'s Gambit** isn\'t covered in our course FAQ materials. However, I can answer based on general knowledge!\n\nThe **Queen\'s Gambit** is a classic chess opening that begins with the moves:\n\n1. **d4 d5**\n2. **c4**\n\nHere\'s a quick breakdown:\n\n- **The idea:** White offers a pawn (the one on c4) to Black in exchange for better control of the center of the board. If Black accepts the pawn (the **Queen\'s Gambit Accepted**), White can try to gain a positional advantage. If Black declines (the **Queen\'s Gambit Declined** — usually with **e6**), the game tends to be more closed and strategic.\n- **It\'s not a "true" gambit:** In most cases, White can safely regain the pawn if needed, so it\'s more of a positional opening than a risky sacrifice.\n- **Popularity:** It\'s one of the oldest and most respected openings in chess, used at all levels from beginners to world champions.\n- **Pop culture:** It\'s also the title of a famous Netflix miniseries about a che

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query": "Queen's Gambit"}
iteration #2...
function_call: search {"query": "queen gambit"}
iteration #3...
ASSISTANT:
Based on my search of the course FAQ database, I couldn't find any information related to the "Queen's Gambit" in the context of this course.

This appears to be an off-topic question not related to the course material or logistics. The Queen's Gambit is a well-known chess opening (1.d4 d5 2.c4) and also the title of a popular Netflix miniseries, but it doesn't seem to be connected to anything covered in this course.

If you have a question about the course itself — assignments, lecture content, deadlines, grading, or any other logistics — I'd be happy to help! Is there anything course-related you'd like to explore?


'Based on my search of the course FAQ database, I couldn\'t find any information related to the "Queen\'s Gambit" in the context of this course.\n\nThis appears to be an off-topic question not related to the course material or logistics. The Queen\'s Gambit is a well-known chess opening (1.d4 d5 2.c4) and also the title of a popular Netflix miniseries, but it doesn\'t seem to be connected to anything covered in this course.\n\nIf you have a question about the course itself — assignments, lecture content, deadlines, grading, or any other logistics — I\'d be happy to help! Is there anything course-related you\'d like to explore?'